# Trying out `seesaw.interp` and `seesaw.pathways`

Runs every technique in `seesaw.interp` and the `circuit_discovery` pathway on IOI, with sanity checks that should hold for **any** weights. Each check prints `PASS` or `FAIL`.

- `TINY = False` uses GPT-2 small, so the heads it finds can be compared with Wang et al. (2022).
- `TINY = True` uses a 2-layer random model, which runs in seconds and checks only the mechanics.

`autoreload` is on, so after editing `seesaw/interp` or `seesaw/pathways`, re-run only the cell for that technique.

In [ ]:
%load_ext autoreload
%autoreload 2

from functools import partial
from pathlib import Path

import torch
from IPython.display import Image, display
from transformer_lens import HookedTransformer, HookedTransformerConfig

from seesaw.interp._helpers import get_logit_diff, sweep_chunks, tokens_to_ids
from seesaw.interp.causal_interventions import ablation, activation_patching
from seesaw.interp.visualization import attention_pattern, direct_logit_attribution, logit_lens
from seesaw.pathways import circuit_discovery

# The seesaw modules call matplotlib.use("Agg") on import; switch back to inline.
%matplotlib inline

failures = []

def show(result):
    for p in result.plot_paths:
        display(Image(filename=str(p)))

def check(name, ok, detail=""):
    print(f"[{'PASS' if ok else 'FAIL'}] {name}" + (f"  ({detail})" if detail else ""))
    if not ok:
        failures.append(name)

## Model and prompts

Clean and corrupted prompts differ only at S2, so they tokenize to the same length.

In [ ]:
TINY = False

if TINY:
    cfg = HookedTransformerConfig(
        n_layers=2, d_model=32, d_head=8, n_heads=4, d_mlp=64, n_ctx=64,
        act_fn="gelu", tokenizer_name="gpt2", normalization_type="LN", seed=0)
    model = HookedTransformer(cfg).eval()
else:
    model = HookedTransformer.from_pretrained("gpt2")

PROMPTS   = ["When Mary and John went to the store, John gave a drink to",
             "When Alice and Bob went to the park, Bob gave a ball to"]
CORRUPTED = ["When Mary and John went to the store, Mary gave a drink to",
             "When Alice and Bob went to the park, Alice gave a ball to"]
POS, NEG  = [" Mary", " Alice"], [" John", " Bob"]

OUT = Path("outputs/notebook")
L, H = model.cfg.n_layers, model.cfg.n_heads
tokens = model.to_tokens(PROMPTS)
io_ids, s_ids = tokens_to_ids(model, POS), tokens_to_ids(model, NEG)
print(model.cfg.model_name, L, "layers ×", H, "heads")

## Helpers

`sweep_chunks` decides how ablation and patching batch their interventions. An off-by-one here silently attributes results to the wrong head or position.

In [ ]:
for n, p, r in [(12, 2, 6), (7, 3, 6), (3, 100, 64)]:
    chunks = list(sweep_chunks(n, p, r))
    covered = [i for a, b in chunks for i in range(a, b)]
    check(f"sweep_chunks({n},{p},{r}) covers each index once", covered == list(range(n)), chunks)

## Logit lens

The final layer's lens has to equal the model's own prediction.

In [ ]:
ll = logit_lens.compute(model, PROMPTS, OUT)
for layer in ll.data["layers"]:
    print(f"L{layer['layer']:>2}", layer["top_tokens"][:3])

with torch.no_grad():
    model_top = model.to_string(model(tokens)[0, -1].argmax().item())
check("final layer == model prediction", ll.data["final_top_token"] == model_top,
      f"{ll.data['final_top_token']!r} vs {model_top!r}")
show(ll)

## Attention patterns

In [ ]:
ap = attention_pattern.compute(model, PROMPTS, OUT, layers=[L - 1])
rows = torch.tensor(ap.data[f"layer_{L-1}"]["final_token_attn"])
check("attention rows sum to 1", torch.allclose(rows.sum(-1), torch.ones(H), atol=1e-4))
show(ap)

## Direct logit attribution

On GPT-2 small, expect the name movers L9H9, L9H6 and L10H0 near the top, and L10H7 and L11H10 negative.

In [ ]:
dla = direct_logit_attribution.compute(model, PROMPTS, POS, NEG, OUT)
print(dla.data["top_heads"])
mags = [abs(v) for _, v in dla.data["top_heads"]]
check("top_heads sorted by magnitude", mags == sorted(mags, reverse=True))
show(dla)

## Causal interventions

Descriptive tools show what a component writes; these test whether it's load-bearing.

### Ablation (zero)

The batched sweep must match ablating one head at a time by hand, with or without small chunks.

In [ ]:
abl = ablation.compute(model, PROMPTS, POS, NEG, OUT, ablation_type="zero")
print("baseline:", abl.data["baseline_ld"], " top:", abl.data["top_heads"])

effects  = torch.tensor(abl.data["effects"])
baseline = get_logit_diff(model, tokens, io_ids, s_ids)
check("baseline_ld == get_logit_diff", abs(abl.data["baseline_ld"] - baseline) < 1e-3)

worst = 0.0
for layer in range(L):
    for head in range(H):
        def zero(v, hook, h=head):
            v = v.clone(); v[:, :, h, :] = 0.0; return v
        with model.hooks(fwd_hooks=[(f"blocks.{layer}.attn.hook_z", zero)]):
            naive = baseline - get_logit_diff(model, tokens, io_ids, s_ids)
        worst = max(worst, abs(naive - effects[layer, head].item()))
check("batched == naive per-head loop", worst < 1e-3, f"max err {worst:.2e}")
show(abl)

In [ ]:
# Force small chunks to exercise the chunk boundaries.
orig = ablation.sweep_chunks
ablation.sweep_chunks = partial(sweep_chunks, max_rows=3 * len(PROMPTS))
try:
    abl_chunked = ablation.compute(model, PROMPTS, POS, NEG, OUT, ablation_type="zero")
finally:
    ablation.sweep_chunks = orig
diff = (torch.tensor(abl_chunked.data["effects"]) - effects).abs().max().item()
check("chunked == unchunked", diff < 1e-4, f"max err {diff:.2e}")

### Ablation (mean)

This is what the `circuit_discovery` pathway falls back to when it has no corrupted prompts. Each head's `z` is replaced by its mean over batch and sequence. It must match the per-head loop too.

In [ ]:
abl_mean = ablation.compute(model, PROMPTS, POS, NEG, OUT, ablation_type="mean")
print("top:", abl_mean.data["top_heads"])

with torch.no_grad():
    _, cache = model.run_with_cache(tokens, names_filter=lambda n: n.endswith("hook_z"))
mean_effects = torch.tensor(abl_mean.data["effects"])

worst = 0.0
for layer in range(L):
    mean_z = cache["z", layer].mean(dim=[0, 1])   # [n_heads, d_head]
    for head in range(H):
        def mean_ablate(v, hook, h=head, m=mean_z):
            v = v.clone(); v[:, :, h, :] = m[h]; return v
        with model.hooks(fwd_hooks=[(f"blocks.{layer}.attn.hook_z", mean_ablate)]):
            naive = baseline - get_logit_diff(model, tokens, io_ids, s_ids)
        worst = max(worst, abs(naive - mean_effects[layer, head].item()))
check("mean ablation: batched == naive per-head loop", worst < 1e-3, f"max err {worst:.2e}")
show(abl_mean)

### Activation patching

- Patching the final layer's residual at the last position *is* the clean run, so recovery must be 1.0.
- Before the first differing token, clean and corrupt residuals are identical, so recovery must be 0.
- Identical clean and corrupt prompts must give zeros, not NaN.

In [ ]:
patch = activation_patching.compute(model, PROMPTS, CORRUPTED, POS, NEG, OUT)
eff = torch.tensor(patch.data["patch_effects"])
print("clean LD", patch.data["clean_ld"], " corrupt LD", patch.data["corrupt_ld"])
print(patch.data["top_recovery_positions"])

check("final layer, last pos recovers 1.0", abs(eff[-1, -1].item() - 1) < 1e-3, f"{eff[-1,-1].item():.4f}")
first_diff = (tokens != model.to_tokens(CORRUPTED)).any(0).nonzero()[0].item()
check(f"positions < {first_diff} recover 0", eff[:, :first_diff].abs().max().item() < 1e-4)

same = torch.tensor(activation_patching.compute(model, PROMPTS, PROMPTS, POS, NEG, OUT).data["patch_effects"])
check("clean == corrupt gives zeros, not NaN", not same.isnan().any() and same.abs().max().item() == 0)
show(patch)

## Pathway: circuit discovery

In [ ]:
pw = circuit_discovery.run(model, PROMPTS, POS, NEG, OUT / "pathway",
                           corrupted_prompts=CORRUPTED)
print(pw.to_markdown())

f = pw.findings
check("measure → nominate → verify order",
      [s for s in pw.steps][:3] == ["direct_logit_attribution", "attribution_patching", "head_patching"]
      or pw.stopped_early is not None, list(pw.steps))
check("confirmed writers ∩ writes_but_redundant = ∅",
      not set(f.get("confirmed_writers", [])) & set(f.get("writes_but_redundant", [])))
check("necessary_but_indirect ⊆ confirmed and excludes DLA writers",
      set(f.get("necessary_but_indirect", [])) <= set(f.get("confirmed_heads", []))
      and not set(f.get("necessary_but_indirect", [])) & set(f.get("writes_against_answer", [])))
check("verdict set", pw.verdict is not None, pw.verdict)

no_patch = circuit_discovery.run(model, PROMPTS, POS, NEG, OUT / "pathway_np")
check("no corrupted prompts → ablation sweep instead of patching",
      "head_patching" not in no_patch.steps
      and ("ablation_sweep" in no_patch.steps or no_patch.stopped_early is not None))

## Summary

In [ ]:
print("ALL CHECKS PASSED" if not failures else f"{len(failures)} FAILED: {failures}")